## ERP Analysis — Multi-Participant

**Instructions:**
- In the *File Loading* cell below, add one `(eeg_path, beh_path)` tuple per participant.
- `N_PARTICIPANTS` is counted automatically — you don't need to set it.
- Column name notes:
  - `"Stimulus"` — event-type column in the PsychoPy CSV
  - `"Marker Timestamp"` — event-time column in the PsychoPy CSV
  - Change these in the `ERPRecording` class if your files use different names.
- If your file path has backslashes where spaces are (pasted from terminal), **remove the backslashes**.

In [ ]:
pip install mne seaborn bokeh

In [ ]:
import pandas as pd
import numpy as np
import mne
import matplotlib.pyplot as plt
import seaborn as sns
from bokeh.plotting import output_notebook
sns.set_style("darkgrid")
output_notebook()

In [ ]:
class ERPRecording:
    """Processes a single participant's EEG + behavioural data into per-condition ERPs."""

    def __init__(self, eeg_df, behav_df, tmin=-0.2, tmax=1.0, sfreq=256):
        self.sfreq = sfreq
        self.tmin = tmin
        self.tmax = tmax
        self.erp_dict = {}       # label -> list of Evoked (one entry per participant when used solo)
        self.grand_avg_dict = {} # label -> single Evoked for this recording

        stim_labels = behav_df["Stimulus"].unique()
        self.event_dict = {label: i + 1 for i, label in enumerate(stim_labels)}

        # Build marker channel
        marker_series = pd.Series(0, index=eeg_df.index)
        for i, row in behav_df.iterrows():
            ts = row["marker_ts"]
            idx = (np.abs(eeg_df["timestamps"] - ts)).idxmin()
            marker_series.iloc[idx] = self.event_dict[row["Stimulus"]]

        # Build MNE Raw
        channels = eeg_df.drop(columns=["timestamps"]).columns.tolist()
        eeg_data = eeg_df.drop(columns="timestamps").T.values
        eeg_data = np.vstack([eeg_data, marker_series.values[np.newaxis, :]])

        info = mne.create_info(
            ch_names=channels + ["markers"],
            sfreq=sfreq,
            ch_types=["eeg"] * len(channels) + ["stim"]
        )
        raw = mne.io.RawArray(eeg_data, info)
        raw.filter(1, 30)

        # Epochs
        events = mne.find_events(raw)
        epochs = mne.Epochs(raw, events, event_id=self.event_dict,
                            tmin=tmin, tmax=tmax, preload=True)

        # Store ERPs
        for label in stim_labels:
            evoked = epochs[label].average()
            self.erp_dict[label] = [evoked]
            self.grand_avg_dict[label] = evoked

    def plot(self, electrode="AF8", markers=None):
        if markers is None:
            markers = list(self.erp_dict.keys())
        for label in markers:
            df = self.grand_avg_dict[label].to_data_frame()
            if electrode not in df.columns:
                print(f"⚠️ Electrode '{electrode}' not found in ERP data.")
                continue
            plt.plot(df["time"], df[electrode], label=label)
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude (μV)")
        plt.title(f"ERP at {electrode}")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()


class GrandAverageERP:
    """
    Combines multiple ERPRecording objects into a grand average across participants.

    Usage:
        grand = GrandAverageERP(sessions)   # sessions is a list of ERPRecording objects
        grand.plot(electrode="AF7")
    """

    def __init__(self, recordings: list):
        if not recordings:
            raise ValueError("No ERPRecording objects provided.")

        self.n = len(recordings)
        print(f"Building grand average from {self.n} participant(s)...")

        # Collect all condition labels seen across participants
        all_labels = set()
        for rec in recordings:
            all_labels.update(rec.grand_avg_dict.keys())

        self.grand_avg_dict = {}
        self.n_per_condition = {}

        for label in all_labels:
            evokeds = [
                rec.grand_avg_dict[label]
                for rec in recordings
                if label in rec.grand_avg_dict
            ]
            self.n_per_condition[label] = len(evokeds)
            if len(evokeds) == 1:
                self.grand_avg_dict[label] = evokeds[0]
            else:
                self.grand_avg_dict[label] = mne.grand_average(evokeds)
            print(f"  '{label}': averaged {len(evokeds)} participant(s)")

    def plot(self, electrode="AF8", markers=None, title_suffix=""):
        if markers is None:
            markers = list(self.grand_avg_dict.keys())

        fig, ax = plt.subplots()
        for label in markers:
            if label not in self.grand_avg_dict:
                print(f"⚠️ Condition '{label}' not found in grand average.")
                continue
            df = self.grand_avg_dict[label].to_data_frame()
            if electrode not in df.columns:
                print(f"⚠️ Electrode '{electrode}' not found in ERP data.")
                continue
            n = self.n_per_condition.get(label, "?")
            ax.plot(df["time"], df[electrode], label=f"{label} (n={n})")

        suffix = f" — {title_suffix}" if title_suffix else f" — {self.n} participant(s)"
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Amplitude (μV)")
        ax.set_title(f"Grand Average ERP at {electrode}{suffix}")
        ax.legend()
        ax.grid(True)
        plt.tight_layout()
        plt.show()

## File Loading

Add one `(eeg_path, beh_path)` tuple per participant.
Copy-paste and edit the lines — `N_PARTICIPANTS` is counted automatically.

In [ ]:
# ── ADD / REMOVE PARTICIPANTS HERE ────────────────────────────────────────────
# Each entry is (eeg_filepath, psychopy_filepath)

participant_files = [
    ("/path/to/participant_01_eeg.csv",  "/path/to/participant_01_beh.csv"),   # participant 1
    ("/path/to/participant_02_eeg.csv",  "/path/to/participant_02_beh.csv"),   # participant 2
    # ("/path/to/participant_03_eeg.csv",  "/path/to/participant_03_beh.csv"),  # participant 3
    # ... keep adding lines, one per participant
]

# ── AUTO-COUNTED — do not edit ─────────────────────────────────────────────────
N_PARTICIPANTS = len(participant_files)
print(f"Loaded file paths for {N_PARTICIPANTS} participant(s).")

## Process All Participants

This cell reads and cleans every file pair, creates one `ERPRecording` per participant,
then combines them into a `GrandAverageERP`.

In [ ]:
def load_and_clean(eeg_path, beh_path):
    """Read, clean, and timestamp-align one EEG + behavioural file pair."""
    eeg_df  = pd.read_csv(eeg_path)
    beh_raw = pd.read_csv(beh_path)

    # Ensure timestamps are numeric
    eeg_df['timestamps']           = pd.to_numeric(eeg_df['timestamps'],           errors='coerce')
    beh_raw['Marker Timestamp']    = pd.to_numeric(beh_raw['Marker Timestamp'],    errors='coerce')

    # Drop rows with invalid timestamps
    eeg_df  = eeg_df[eeg_df['timestamps'].notnull()]
    beh_raw = beh_raw[beh_raw['Marker Timestamp'].notnull()]

    # Convert Unix timestamps to datetime
    eeg_df['timestamps']  = pd.to_datetime(eeg_df['timestamps'],        unit='s')
    beh_raw['marker_ts']  = pd.to_datetime(beh_raw['Marker Timestamp'], unit='s')

    return eeg_df, beh_raw


sessions = []

for i, (eeg_path, beh_path) in enumerate(participant_files, start=1):
    print(f"\n── Participant {i}/{N_PARTICIPANTS} ─────────────────────────")
    try:
        eeg_df, beh_raw = load_and_clean(eeg_path, beh_path)
        rec = ERPRecording(eeg_df, beh_raw, tmin=-0.3, tmax=1.0)
        sessions.append(rec)
        print(f"   ✓ Done")
    except Exception as e:
        print(f"   ✗ Skipped — {e}")

print(f"\nSuccessfully processed {len(sessions)}/{N_PARTICIPANTS} participant(s).")

# Build grand average
grand = GrandAverageERP(sessions)

## Grand Average Plots

Each cell below plots the grand average ERP at one electrode across all loaded participants.
The legend shows each condition with the number of participants it was averaged over.

In [ ]:
# Grand average — AF7
grand.plot(electrode="AF7")

In [ ]:
# Grand average — AF8
grand.plot(electrode="AF8")

In [ ]:
# Grand average — TP9
grand.plot(electrode="TP9")

In [ ]:
# Grand average — TP10
grand.plot(electrode="TP10")

## Optional: Plot a Single Participant

If you want to inspect one participant's data before grand averaging, use `sessions[i]` (0-indexed).

In [ ]:
# Change index to inspect a different participant (0 = first, 1 = second, etc.)
PARTICIPANT_INDEX = 0

if PARTICIPANT_INDEX < len(sessions):
    s = sessions[PARTICIPANT_INDEX]
    conditions = list(s.grand_avg_dict.keys())
    for elec in ["AF7", "AF8", "TP9", "TP10"]:
        s.plot(electrode=elec, markers=conditions)
else:
    print(f"Index {PARTICIPANT_INDEX} out of range — only {len(sessions)} session(s) loaded.")